### Load the dataset

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

folder_path = "UTKFace"
data = []

for filename in sorted(os.listdir(folder_path)):
    

    if not filename.endswith(".jpg"):
        continue
    try:
        # Extract age from filename
        age = int(filename.split("_")[0])
        img_path = os.path.join(folder_path, filename)
        # Append to list
        data.append([img_path, age])

    except Exception as e:
        print(f"Skipping {filename} due to error: {e}")

# Create DataFrame
df = pd.DataFrame(data, columns=["image_path", "age"])

print("Number of samples:", len(df))
df

Number of samples: 23708


,image_path,age
0,UTKFace\100_0_0_20170112213500903.jpg.chip.jpg,100
1,UTKFace\100_0_0_20170112215240346.jpg.chip.jpg,100
2,UTKFace\100_1_0_20170110183726390.jpg.chip.jpg,100
3,UTKFace\100_1_0_20170112213001988.jpg.chip.jpg,100
4,UTKFace\100_1_0_20170112213303693.jpg.chip.jpg,100
...,...,...
23703,UTKFace\9_1_3_20161220222856346.jpg.chip.jpg,9
23704,UTKFace\9_1_3_20170104222949455.jpg.chip.jpg,9
23705,UTKFace\9_1_4_20170103200637399.jpg.chip.jpg,9
23706,UTKFace\9_1_4_20170103200814791.jpg.chip.jpg,9


In [3]:
import torch
print(torch.cuda.is_available())        # Should be True
print(torch.cuda.get_device_name(0))    # Should show RTX 5060
print(torch.__version__)                # Should be 2.7+


ModuleNotFoundError: No module named 'torch'

### Check age distribution and class imbalance

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic stats
print(df["age"].describe())

# Age distribution
plt.figure(figsize=(14, 5))

# Plot 1 - histogram
plt.subplot(1, 2, 1)
sns.histplot(df["age"], bins=50, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")

# Plot 2 - by age group (binned)
plt.subplot(1, 2, 2)
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 120]
labels = ["0-10", "11-20", "21-30", "31-40", "41-50", "51-60", "61-70", "71-80", "81-90", "90+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)
df["age_group"].value_counts().sort_index().plot(kind="bar")
plt.title("Samples per Age Group")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [12]:
# Check exact counts for rare groups
print(df["age_group"].value_counts().sort_index())

# Check how many samples exist above 90
print(f"Samples above 90: {len(df[df['age'] > 90])}")
print(f"Max age: {df['age'].max()}")

age_group
0-10     3218
11-20    1659
21-30    7784
31-40    4339
41-50    2100
51-60    2211
61-70    1172
71-80     685
81-90     453
90+        87
Name: count, dtype: int64
Samples above 90: 87
Max age: 116


As 90+ samples are very few, we decide to clip at 90.

In [ ]:
df = df[df["age"] <= 90]
print(len(df))

23621


To handle class imbalance, we will apply oversampling with data augmentation for minority classes, as well as weighted loss.

In [14]:
df['age'].max()

np.int64(90)